# DeepPredict: 02. Feature Engineering & Preprocessing

This notebook demonstrates:
1. Engine-level Train/Validation splitting (ensuring zero cross-engine sequence leakage).
2. Fitting MinMaxScaler strictly on training engines.
3. Generating 30-cycle temporal sliding windows $(N, L, F)$ for PyTorch deep learning models.
4. Extracting summary statistical features (mean, std, min, max, delta) for tree-based baselines.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DatasetConfig, PreprocessingConfig, TARGET_CAPPED_COLUMN
from src.data_loader import load_and_prepare_cmapss
from src.preprocessing import split_train_val_engines, CMapssPreprocessor, create_sliding_sequences
from src.feature_engineering import extract_tabular_window_features

%matplotlib inline

## 1. Load C-MAPSS and Inspect RUL Capping

In [ ]:
train_df, test_df, test_rul_truth, meta = load_and_prepare_cmapss(DatasetConfig(rul_cap=125))
print(f"Loaded {len(train_df)} train rows, {len(test_df)} test rows.")

# Compare uncapped vs capped RUL for Engine 1
e1 = train_df[train_df["unit_number"] == 1]
plt.figure(figsize=(9, 4))
plt.plot(e1["time_in_cycles"], e1["RUL"], "b--", label="Uncapped Linear RUL")
plt.plot(e1["time_in_cycles"], e1[TARGET_CAPPED_COLUMN], "r-", lw=2, label="Piecewise Linear Capped RUL (125)")
plt.title("Target Engineering: Piecewise Linear RUL", fontsize=13, fontweight="bold")
plt.xlabel("Cycles")
plt.ylabel("Target RUL")
plt.legend()
plt.show()

## 2. Zero-Leakage Engine Splitting & Scaler Fitting

In [ ]:
train_split, val_split, tr_ids, val_ids = split_train_val_engines(train_df, val_ratio=0.20, random_seed=42)
print(f"Train Engines: {len(tr_ids)} | Validation Engines: {len(val_ids)}")

preprocessor = CMapssPreprocessor(PreprocessingConfig(window_length=30))
preprocessor.fit(train_split)

print(f"Selected {len(preprocessor.selected_features)} dynamic features:")
print(preprocessor.selected_features)

## 3. Sliding Sequence Window Generation

In [ ]:
train_scaled = preprocessor.transform(train_split)
val_scaled = preprocessor.transform(val_split)

X_train, y_train, _ = create_sliding_sequences(
    train_scaled, preprocessor.selected_features, TARGET_CAPPED_COLUMN, window_length=30
)
X_val, y_val, _ = create_sliding_sequences(
    val_scaled, preprocessor.selected_features, TARGET_CAPPED_COLUMN, window_length=30
)

print(f"X_train sequence tensor shape: {X_train.shape}")
print(f"y_train target shape: {y_train.shape}")
print(f"X_val sequence tensor shape: {X_val.shape}")

## 4. Tabular Feature Extraction for Tree Baselines

In [ ]:
tab_features = extract_tabular_window_features(X_train[:100], preprocessor.selected_features)
print(f"Engineered tabular shape: {tab_features.shape}")
tab_features.head()